In [24]:
import pandas as pd 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score 
from docx import Document
import os
all_results = []

c=6
for s in [1,2,3,4,5,6,7,8,9,10] :
    df_train=pd.read_csv(f"all_files/Train/Dataset_S{s}_C{c}_features.csv") 
    df_test=pd.read_csv(f"all_files/Test/Test_Dataset_S{s}_C{c}_features.csv")
    print("Original training samples:", len(df_train))
    data_train = df_train.sample(frac=1,random_state=42).reset_index(drop=True)
    data_test=df_test
    print("Test samples:", len(df_test))
    #feature_types = ("_peak_to_peak")
    #channels_types = ("Ch1","Ch2","Ch4")
    feature_names = { 1: "all_features_with Hj", 2: "all_features",3:"wave_length_only",4:"Hudgins",5:"MAV_only"}    
    feature_sets = {1: ("_MAV","_RMS", "_iemg", "_ssc", "_WL","_ZC","_variance", "Hjorth_variance","Hjorth_mobility","Hjorth_complexity"), 
                    2: ("_MAV","_RMS", "_iemg", "_variance","_ssc","ZC","_WL") ,3:("_WL",),
                    4:("_ZC", "_MAV","_ssc","_WL"),5:("_MAV")}
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
    data = {"Model": [], "MAE": [],  "RMSE": [], "R2": []}
    results = []
    for i, features in feature_sets.items():    
        feature_columns = [
           col for col in data_train.columns
            if col.endswith(features)]   
        X_train = data_train[feature_columns]
        X_test = data_test[feature_columns]
        y_train = data_train["MVC_Percent"]
        y_test = data_test["MVC_Percent"]
        
        #Extra trees

        name = "Extra trees"
        model_et = ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1 )
        model_et.fit(X_train, y_train)
        y_pred_et = model_et.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred_et)
        rmse = mean_squared_error(y_test, y_pred_et) ** 0.5
        r2 = r2_score(y_test, y_pred_et)
        data["Model"].append(f"{name} - {feature_names[i]}")
        data["MAE"].append(mae)
        data["RMSE"].append(rmse)
        data["R2"].append(r2)
    
        #knn
        name = "knn"
        model_knn = make_pipeline( StandardScaler(), KNeighborsRegressor(n_neighbors=5) )
        model_knn.fit(X_train, y_train)
        y_pred_knn = model_knn.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred_knn)
        rmse = mean_squared_error(y_test, y_pred_knn) ** 0.5
        r2 = r2_score(y_test, y_pred_knn)
        data["Model"].append(f"{name} - {feature_names[i]}")
        data["MAE"].append(mae)
        data["RMSE"].append(rmse)
        data["R2"].append(r2)

        # ridge regression
        name = "ridge regression"
        model_ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0) )
        model_ridge.fit(X_train, y_train) 
        y_pred_ridge = model_ridge.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred_ridge)
        rmse = mean_squared_error(y_test, y_pred_ridge) ** 0.5
        r2 = r2_score(y_test, y_pred_ridge)
        data["Model"].append(f"{name} - {feature_names[i]}")
        data["MAE"].append(mae)
        data["RMSE"].append(rmse)
        data["R2"].append(r2)
        
    df = pd.DataFrame(data)
    df["Subject"] = f"S{s}"
    df["Class"] = c
    all_results.append(df.copy())
    df = df.sort_values("MAE")
    best_algorithm = df.iloc[0]["Model"]
    print(f"S{s}"+ best_algorithm)
    print(df)

    # save data in word file 
    word_path = "all_files/Results_new.docx"
    # Create a new Word file if it doesn't exist;
    # otherwise, open the existing file
    if os.path.exists(word_path):
        doc = Document(word_path)
    else:
        doc = Document()
        doc.add_heading("EMG Regression Results", level=1)
    
    # Title changes according to subject and feature set
    doc.add_heading(f"Subject S{s} - Class{c} ", level=2)
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"
    
    for cell, name in zip(table.rows[0].cells, df.columns):
        cell.text = name
    
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for cell, value in zip(cells, row):
            cell.text = (f"{value:.4f}"
                if isinstance(value, (int, float, np.number))
                else str(value) )
    doc.save(word_path)

# Combine results from all subjects
final_results = pd.concat(all_results, ignore_index=True)

# Calculate average metrics for each algorithm and feature set
average_results = (
    final_results
    .groupby("Model")[["MAE", "RMSE", "R2"]]
    .mean()
    .reset_index()
)

# Calculate standard deviation
std_results = (
    final_results
    .groupby("Model")[["MAE", "RMSE", "R2"]]
    .std()
    .reset_index()
)

# Sort by Mean MAE
average_results_mae= average_results.sort_values("MAE")
average_results_rmse= average_results.sort_values("RMSE")
average_results_R2= average_results.sort_values("R2")

print("\nAverage results across 10 subjects:")
print(average_results_mae)
best_MAE = average_results_mae.iloc[0]["Model"]
best_RMSE = average_results_rmse.iloc[0]["Model"]
best_R2 = average_results_R2.iloc[-1]["Model"]

print("\nStandard deviation across subjects:")
print(std_results)

# Best overall algorithm and feature set

print("\nBest MAE for :", best_MAE)
print("\nBest RMSE for:", best_RMSE)
print("\nBest R2 for :", best_R2)
doc = Document(word_path)

doc.add_heading(
    f"Average Results Across 10 Subjects - Class {c}",
    level=2
)

table = doc.add_table(rows=1, cols=4)
table.style = "Table Grid"

for cell, name in zip(table.rows[0].cells, average_results.columns):
    cell.text = name

for _, row in average_results_mae.iterrows():
    cells = table.add_row().cells

    for cell, value in zip(cells, row):
        cell.text = (
            f"{value:.4f}"
            if isinstance(value, (int, float, np.number))
            else str(value)
        )

doc.save(word_path)


Original training samples: 249
Test samples: 84
S1ridge regression - MAV_only
                                      Model        MAE       RMSE        R2  \
14              ridge regression - MAV_only  17.868464  21.328337 -0.137245   
8       ridge regression - wave_length_only  19.964748  22.041111 -0.214526   
12                   Extra trees - MAV_only  20.519048  23.816618 -0.418078   
13                           knn - MAV_only  23.880952  27.831979 -0.936548   
5           ridge regression - all_features  23.922647  29.508848 -1.176930   
0        Extra trees - all_features_with Hj  24.272619  27.587610 -0.902691   
11               ridge regression - Hudgins  24.545145  29.067286 -1.112268   
3                Extra trees - all_features  24.983929  28.278158 -0.999135   
9                     Extra trees - Hudgins  25.552976  28.867952 -1.083397   
2   ridge regression - all_features_with Hj  25.594220  31.396075 -1.464284   
6            Extra trees - wave_length_only  25.84583